In [ ]:
from pathlib import Path

mmash_candidates = list((RAW_DIR / 'mmash').rglob('DataPaper'))

print("Found MMASH candidates:")
for p in mmash_candidates:
    print(p)

if not mmash_candidates:
    raise FileNotFoundError(
        'DataPaper not found inside data/raw/mmash. '
        'Move the unpacked MMASH folder there first.'
    )

MMASH_BASE = mmash_candidates[0]
print("MMASH_BASE =", MMASH_BASE)

user_dirs = sorted([p for p in MMASH_BASE.iterdir() if p.is_dir() and p.name.startswith('user_')])

print(f'Found {len(user_dirs)} users')
print([p.name for p in user_dirs[:10]])


In [ ]:
sleep_rows = []
questionnaire_rows = []
user_info_rows = []

for user_path in user_dirs:
    sleep_rows.append(build_sleep_row(user_path))
    questionnaire_rows.append(build_questionnaire_row(user_path))
    user_info_rows.append(build_user_info_row(user_path))

sleep_df = pd.concat(sleep_rows, ignore_index=True)
questionnaire_df = pd.concat(questionnaire_rows, ignore_index=True)
user_info_df = pd.concat(user_info_rows, ignore_index=True)

mmash_df = sleep_df.merge(questionnaire_df, on='user_id', how='inner')
mmash_df = mmash_df.merge(user_info_df, on='user_id', how='inner')

print(mmash_df.shape)
mmash_df.head() 

In [ ]:
ZIP_PATH = Path('Documents\mash')
ZIP_PATH.exists(), ZIP_PATH

In [ ]:
ZIP_PATH = Path('../data/raw/MMASH.zip')

In [ ]:
zf = zipfile.ZipFile(ZIP_PATH)
names = zf.namelist()

user_dirs = sorted(set(
    re.findall(r'DataPaper/(user_\d+)/', name)[0]
    for name in names
    if re.findall(r'DataPaper/(user_\d+)/', name)
))

print(f'Found {len(user_dirs)} users')
print(user_dirs[:10])

In [ ]:
import os
from pathlib import Path

print("Current working directory:", os.getcwd())
print("\nFiles here:")
for p in Path('.').iterdir():
    print(p.name)

In [ ]:
from pathlib import Path

for p in Path('.').iterdir():
    print(p)

In [ ]:
from pathlib import Path

search_roots = [
    Path(r'C:\Users\vi\Downloads'),
    Path(r'C:\Users\vi\Desktop'),
    Path(r'C:\Users\vi\Documents'),
    Path(r'C:\Users\vi'),
]

matches = []

for root in search_roots:
    if root.exists():
        for p in root.rglob('user_info.csv'):
            matches.append(p)

print(f'Found {len(matches)} matches')
for p in matches[:50]:
    print(p)

In [ ]:
from pathlib import Path

example_file = matches[0]
BASE_DIR = example_file.parent.parent   # ...\DataPaper
print("BASE_DIR =", BASE_DIR)

user_dirs = sorted([p for p in BASE_DIR.iterdir() if p.is_dir() and p.name.startswith('user_')])

print(f'Found {len(user_dirs)} users')
print([p.name for p in user_dirs[:10]])

In [ ]:
import pandas as pd
import numpy as np

def read_csv_from_folder(path):
    return pd.read_csv(path)

In [ ]:
SLEEP_COLS = [
    'Latency',
    'Efficiency',
    'Total Minutes in Bed',
    'Total Sleep Time (TST)',
    'Wake After Sleep Onset (WASO)',
    'Number of Awakenings',
    'Average Awakening Length',
    'Movement Index',
    'Fragmentation Index',
    'Sleep Fragmentation Index'
]

def build_sleep_row(user_path):
    df = read_csv_from_folder(user_path / 'sleep.csv')

    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])

    if df.empty:
        row = {'user_id': user_path.name, 'n_sleep_rows': 0}
        for col in SLEEP_COLS:
            row[col] = np.nan
        return pd.DataFrame([row])

    for col in SLEEP_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    existing_cols = [c for c in SLEEP_COLS if c in df.columns]
    agg = df[existing_cols].mean(numeric_only=True).to_dict()
    agg['user_id'] = user_path.name
    agg['n_sleep_rows'] = len(df)

    return pd.DataFrame([agg])

In [ ]:
def build_questionnaire_row(user_path):
    df = read_csv_from_folder(user_path / 'questionnaire.csv')

    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])

    row = df.iloc[0].to_dict()
    row['user_id'] = user_path.name

    return pd.DataFrame([row])

In [ ]:
def build_user_info_row(user_path):
    df = read_csv_from_folder(user_path / 'user_info.csv')

    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])

    row = df.iloc[0].to_dict()
    row['user_id'] = user_path.name

    return pd.DataFrame([row])

In [ ]:
sleep_rows = []
questionnaire_rows = []
user_info_rows = []

for user_path in user_dirs:
    sleep_rows.append(build_sleep_row(user_path))
    questionnaire_rows.append(build_questionnaire_row(user_path))
    user_info_rows.append(build_user_info_row(user_path))

sleep_df = pd.concat(sleep_rows, ignore_index=True)
questionnaire_df = pd.concat(questionnaire_rows, ignore_index=True)
user_info_df = pd.concat(user_info_rows, ignore_index=True)

mmash_df = sleep_df.merge(questionnaire_df, on='user_id', how='inner')
mmash_df = mmash_df.merge(user_info_df, on='user_id', how='inner')

print(mmash_df.shape)
mmash_df.head()

In [ ]:
mmash_df.to_csv(PROCESSED_DIR / 'mmash_modeling_dataset.csv', index=False)
print('Saved: mmash_modeling_dataset.csv')

In [ ]:
import pandas as pd

mmash_df = pd.read_csv(PROCESSED_DIR / 'mmash_modeling_dataset.csv')
print(mmash_df.shape)
mmash_df.head()

In [ ]:
mmash_df.columns.tolist()

In [ ]:
mmash_df.isna().mean().sort_values(ascending=False)

In [ ]:
target_col = 'Daily_stress'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

target_col = 'Daily_stress'

feature_cols = [
    'Latency',
    'Efficiency',
    'Total Minutes in Bed',
    'Total Sleep Time (TST)',
    'Wake After Sleep Onset (WASO)',
    'Number of Awakenings',
    'Average Awakening Length',
    'Movement Index',
    'Fragmentation Index',
    'Sleep Fragmentation Index',
    'Age',
    'Weight',
    'Height',
    'Gender'
]

X = mmash_df[feature_cols].copy()
y = mmash_df[target_col].copy()

categorical_features = ['Gender']
numeric_features = [c for c in feature_cols if c not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

ridge_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge())
])

ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)

print('Ridge MAE:', mean_absolute_error(y_test, ridge_pred))
print('Ridge R2:', r2_score(y_test, ridge_pred))

rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print('RF MAE:', mean_absolute_error(y_test, rf_pred))
print('RF R2:', r2_score(y_test, rf_pred))

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_mae = -cross_val_score(
    rf_model,
    X,
    y,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

cv_r2 = cross_val_score(
    rf_model,
    X,
    y,
    cv=cv,
    scoring='r2'
)

print('CV MAE:', cv_mae)
print('Mean CV MAE:', cv_mae.mean())

print('CV R2:', cv_r2)
print('Mean CV R2:', cv_r2.mean())

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, rf_pred, alpha=0.7)
plt.xlabel(f'Actual {target_col}')
plt.ylabel(f'Predicted {target_col}')
plt.title(f'MMASH Prediction: {target_col}')
plt.grid(alpha=0.3)

min_val = min(y_test.min(), rf_pred.min())
max_val = max(y_test.max(), rf_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
corr_cols = [
    'Latency',
    'Efficiency',
    'Total Sleep Time (TST)',
    'Wake After Sleep Onset (WASO)',
    'Number of Awakenings',
    'Fragmentation Index',
    'Sleep Fragmentation Index',
    'Daily_stress',
    'STAI1',
    'STAI2',
    'Pittsburgh'
]

corr = mmash_df[corr_cols].corr(numeric_only=True)
corr['Daily_stress'].sort_values(ascending=False)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.scatter(mmash_df['Fragmentation Index'], mmash_df['Daily_stress'], alpha=0.7)
plt.xlabel('Fragmentation Index')
plt.ylabel('Daily_stress')
plt.title('Daily Stress vs Fragmentation Index')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.scatter(mmash_df['Sleep Fragmentation Index'], mmash_df['Daily_stress'], alpha=0.7)
plt.xlabel('Sleep Fragmentation Index')
plt.ylabel('Daily_stress')
plt.title('Daily Stress vs Sleep Fragmentation Index')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Preliminary correlation analysis suggests that higher sleep fragmentation is associated with higher daily stress in MMASH participants. Fragmentation-related metrics showed stronger associations with stress than total sleep time, efficiency, or latency

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

target_col = 'Daily_stress'

feature_cols = [
    'Latency',
    'Efficiency',
    'Total Minutes in Bed',
    'Total Sleep Time (TST)',
    'Wake After Sleep Onset (WASO)',
    'Number of Awakenings',
    'Average Awakening Length',
    'Movement Index',
    'Fragmentation Index',
    'Sleep Fragmentation Index',
    'Age',
    'Weight',
    'Height',
    'Gender'
]

X = mmash_df[feature_cols].copy()
y = mmash_df[target_col].copy()

categorical_features = ['Gender']
numeric_features = [c for c in feature_cols if c not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

ridge_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge())
])

ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)

print('Ridge MAE:', mean_absolute_error(y_test, ridge_pred))
print('Ridge R2:', r2_score(y_test, ridge_pred))

rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print('RF MAE:', mean_absolute_error(y_test, rf_pred))
print('RF R2:', r2_score(y_test, rf_pred))

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_mae = -cross_val_score(
    rf_model,
    X,
    y,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

cv_r2 = cross_val_score(
    rf_model,
    X,
    y,
    cv=cv,
    scoring='r2'
)

print('CV MAE:', cv_mae)
print('Mean CV MAE:', cv_mae.mean())

print('CV R2:', cv_r2)
print('Mean CV R2:', cv_r2.mean())

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, rf_pred, alpha=0.7)
plt.xlabel(f'Actual {target_col}')
plt.ylabel(f'Predicted {target_col}')
plt.title(f'MMASH Prediction: {target_col}')
plt.grid(alpha=0.3)

min_val = min(y_test.min(), rf_pred.min())
max_val = max(y_test.max(), rf_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')

plt.tight_layout()
plt.show()

## Model interpretation

A baseline regression model was trained to predict Daily_stress from sleep-related metrics and basic demographic variables.

Given the small sample size, model performance should be interpreted cautiously. The purpose of this analysis is exploratory: to test whether sleep quality metrics contain predictive signal for stress-related questionnaire outcomes.


Baseline regression models showed weak predictive performance for `Daily_stress` in this MMASH sample.

Although correlation analysis suggested that sleep fragmentation metrics were associated with daily stress, the predictive models did not generalize well. Both Ridge regression and Random Forest produced negative R² values, indicating that the models performed worse than a simple mean-based baseline.

This suggests that, in this small sample, sleep metrics alone do not provide a strong enough signal for robust prediction of daily stress.

## I explored whether sleep quality metrics contain predictive signal for stress-related questionnaire outcomes.
In MMASH, fragmentation-related metrics showed the strongest associations with daily stress, but baseline models did not generalize well, highlighting both the promise and the limitations of small open wearable datasets.

## Conclusion

This MMASH case study suggests that sleep fragmentation metrics may be associated with higher daily stress at the correlation level. However, baseline regression models showed poor generalization performance, with negative R² values across both hold-out and cross-validation settings.

These results indicate that sleep metrics alone are insufficient for robust stress prediction in this small sample. The analysis is therefore best interpreted as an exploratory proof of concept rather than a reliable predictive model.

In [ ]:
for target in ['STAI1', 'Pittsburgh']:
    print(f'\nCorrelations with {target}')
    print(
        mmash_df[
            [
                'Latency',
                'Efficiency',
                'Total Sleep Time (TST)',
                'Wake After Sleep Onset (WASO)',
                'Number of Awakenings',
                'Fragmentation Index',
                'Sleep Fragmentation Index',
                target
            ]
        ].corr(numeric_only=True)[target].sort_values(ascending=False)
    )

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

feature_cols = [
    'Latency',
    'Efficiency',
    'Total Minutes in Bed',
    'Total Sleep Time (TST)',
    'Wake After Sleep Onset (WASO)',
    'Number of Awakenings',
    'Average Awakening Length',
    'Movement Index',
    'Fragmentation Index',
    'Sleep Fragmentation Index',
    'Age',
    'Weight',
    'Height',
    'Gender'
]

categorical_features = ['Gender']
numeric_features = [c for c in feature_cols if c not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features)
    ]
)

targets = ['STAI1', 'Pittsburgh']
results = []

for target_col in targets:
    X = mmash_df[feature_cols].copy()
    y = mmash_df[target_col].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    ridge_model = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', Ridge())
    ])

    ridge_model.fit(X_train, y_train)
    ridge_pred = ridge_model.predict(X_test)

    rf_model = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=2,
            random_state=42
        ))
    ])

    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_test)

    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    cv_mae = -cross_val_score(
        rf_model,
        X,
        y,
        cv=cv,
        scoring='neg_mean_absolute_error'
    )

    cv_r2 = cross_val_score(
        rf_model,
        X,
        y,
        cv=cv,
        scoring='r2'
    )

    results.append({
        'target': target_col,
        'ridge_mae': mean_absolute_error(y_test, ridge_pred),
        'ridge_r2': r2_score(y_test, ridge_pred),
        'rf_mae': mean_absolute_error(y_test, rf_pred),
        'rf_r2': r2_score(y_test, rf_pred),
        'cv_mae_mean': cv_mae.mean(),
        'cv_r2_mean': cv_r2.mean()
    })

    plt.figure(figsize=(5, 5))
    plt.scatter(y_test, rf_pred, alpha=0.7)
    plt.xlabel(f'Actual {target_col}')
    plt.ylabel(f'Predicted {target_col}')
    plt.title(f'RF Prediction: {target_col}')
    plt.grid(alpha=0.3)

    min_val = min(y_test.min(), rf_pred.min())
    max_val = max(y_test.max(), rf_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')
    plt.tight_layout()
    plt.show()

results_df = pd.DataFrame(results)
results_df

## Target comparison interpretation

Among the tested questionnaire outcomes, `STAI1` showed the most promising predictive signal from sleep-related features. The Random Forest baseline achieved a small positive hold-out R², suggesting that sleep metrics may contain limited information about state anxiety in this sample.

In contrast, `Daily_stress` and `Pittsburgh` were not predicted robustly, especially under cross-validation. This indicates that, in MMASH, aggregated sleep features may be more informative for some anxiety-related measures than for broader stress or sleep-quality questionnaire scores.

## Conclusion

This MMASH case study suggests that sleep fragmentation and related sleep-quality metrics show meaningful exploratory associations with stress- and anxiety-related questionnaire outcomes.

At the predictive modeling stage, `STAI1` emerged as the most promising target among those tested, while `Daily_stress` and `Pittsburgh` showed weak generalization performance. Overall, the results support an exploratory interpretation: sleep metrics may contain limited predictive signal for state anxiety, but the small sample size prevents strong conclusions.

In [ ]:
from sklearn.inspection import permutation_importance
import pandas as pd
import matplotlib.pyplot as plt

target_col = 'STAI1'

X = mmash_df[feature_cols].copy()
y = mmash_df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

perm = permutation_importance(
    rf_model,
    X_test,
    y_test,
    n_repeats=20,
    random_state=42,
    scoring='r2'
)

importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance_mean': perm.importances_mean
}).sort_values('importance_mean', ascending=False)

importance_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(importance_df['feature'], importance_df['importance_mean'])
plt.gca().invert_yaxis()
plt.xlabel('Permutation importance')
plt.title('Feature Importance for STAI1 Prediction')
plt.tight_layout()
plt.show()

In [ ]:
mmash_df.to_csv(PROCESSED_DIR / 'mmash_modeling_dataset.csv', index=False)
print('Saved:', PROCESSED_DIR / 'mmash_modeling_dataset.csv')

In [ ]:
mmash_df.to_csv(PROCESSED_DIR / 'mmash_modeling_dataset.csv', index=False)
print('Saved:', PROCESSED_DIR / 'mmash_modeling_dataset.csv')